# Valorisation d’un Put américain dans le modèle CRR

## Introduction

Le but de ce devoir est de calculer le prix d’un **Put américain** dans le modèle binomial de **Cox-Ross-Rubinstein (CRR)**.

Dans le cas européen, on remonte l’arbre uniquement par espérance risque-neutre actualisée.  
Dans le cas américain, il faut en plus comparer, à chaque nœud, la valeur de continuation et la valeur d’exercice immédiat.  
C’est cette comparaison locale qui introduit un opérateur de maximum dans la récurrence rétrograde.

## 1) Rappel du modèle CRR

On fixe une maturité \(T\) et un nombre de pas \(N\).  
Les dates sont
\[
t_n = n\frac{T}{N}, \qquad n=0,\dots,N,
\]
avec \(\Delta t = T/N\).

Dans CRR, entre \(t_n\) et \(t_{n+1}\), le sous-jacent est multiplié par :

- \(u = e^{\sigma\sqrt{\Delta t}}\) (mouvement **up**),
- \(d = e^{-\sigma\sqrt{\Delta t}}\) (mouvement **down**).

Au nœud \((n,i)\) (où \(i\) est le nombre de mouvements up),
\[
S_{n,i} = S_0\,u^i d^{n-i}, \qquad i=0,\dots,n.
\]

Sous la probabilité risque-neutre,
\[
p = \frac{e^{r\Delta t}-d}{u-d},
\]
et on doit avoir \(0<p<1\).

Le payoff du Put est
\[
\Phi(S) = (K-S)^+ = \max(K-S,0).
\]

## 2) Récurrence rétrograde pour le Put américain

À maturité :
\[
V_{N,i} = (K-S_{N,i})^+, \qquad i=0,\dots,N.
\]

Pour \(n=N-1,\dots,0\), la valeur au nœud \((n,i)\) est :
\[
V_{n,i}
=
\max\!\left(
(K-S_{n,i})^+,
 e^{-r\Delta t}\left[(1-p)V_{n+1,i}+pV_{n+1,i+1}\right]
\right).
\]

---

### Écriture matricielle de la continuation

On note :
\[
V_n = (V_{n,0},\dots,V_{n,n})^\top \in \mathbb{R}^{n+1},
\quad
V_{n+1}\in\mathbb{R}^{n+2}.
\]

On définit l’opérateur linéaire \(P_{n+1}\in\mathbb{R}^{(n+1)\times(n+2)}\) par
\[
(P_{n+1}V_{n+1})_i = (1-p)V_{n+1,i} + pV_{n+1,i+1}, \quad i=0,\dots,n.
\]

Donc la valeur de continuation s’écrit :
\[
C_n = e^{-r\Delta t}\,P_{n+1}V_{n+1}.
\]

Enfin,
\[
V_n = \max\!\big(\Phi_n,\;C_n\big),
\]
où \(\Phi_n = ((K-S_{n,0})^+,\dots,(K-S_{n,n})^+)^\top\), et le maximum est pris composante par composante.

In [ ]:
import numpy as np

In [ ]:
def stock_prices_level(S0, u, d, n):
    """
    Renvoie le vecteur des prix au niveau n :
    S_{n,i} = S0 * u^i * d^(n-i), i=0,...,n
    """
    i = np.arange(n + 1)
    return S0 * (u ** i) * (d ** (n - i))


def build_transition_matrix(n, p, use_sparse=False):
    """
    Construit l'opérateur P_{n+1} de taille (n+1, n+2), tel que
    (P_{n+1} @ V_{n+1})[i] = (1-p) * V_{n+1}[i] + p * V_{n+1}[i+1].

    - version dense : ndarray numpy
    - version creuse : CSR scipy.sparse
    """
    if use_sparse:
        try:
            from scipy.sparse import coo_matrix
        except ImportError as exc:
            raise ImportError(
                "scipy est nécessaire pour use_sparse=True. "
                "Installer scipy ou utiliser use_sparse=False."
            ) from exc

        rows = np.repeat(np.arange(n + 1), 2)
        cols = np.empty(2 * (n + 1), dtype=int)
        cols[0::2] = np.arange(n + 1)
        cols[1::2] = np.arange(1, n + 2)

        data = np.empty(2 * (n + 1), dtype=float)
        data[0::2] = 1.0 - p
        data[1::2] = p

        P = coo_matrix((data, (rows, cols)), shape=(n + 1, n + 2))
        return P.tocsr()

    # Version dense
    P = np.zeros((n + 1, n + 2), dtype=float)
    idx = np.arange(n + 1)
    P[idx, idx] = 1.0 - p
    P[idx, idx + 1] = p
    return P

In [ ]:
def price_american_put_crr(S0, K, r, sigma, T, N, use_sparse=False):
    """
    Prix d'un Put américain dans le modèle CRR par récurrence rétrograde.
    La continuation est calculée via produit matrice-vecteur :
    e^{-r dt} * (P_{n+1} @ V_{n+1}).
    """
    if N <= 0:
        raise ValueError("N doit être un entier strictement positif.")

    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = np.exp(-sigma * np.sqrt(dt))
    p = (np.exp(r * dt) - d) / (u - d)

    # Condition d'absence d'arbitrage dans ce schéma
    if not (0.0 < p < 1.0):
        raise ValueError(
            "La probabilité risque-neutre p doit vérifier 0 < p < 1. "
            "Vérifier les paramètres (r, sigma, T, N)."
        )

    discount = np.exp(-r * dt)

    # Condition terminale : V_N = payoff du put
    S_N = stock_prices_level(S0, u, d, N)        # taille N+1
    V = np.maximum(K - S_N, 0.0)                 # taille N+1

    # Récurrence rétrograde
    for n in range(N - 1, -1, -1):
        S_n = stock_prices_level(S0, u, d, n)    # taille n+1
        exercise_now = np.maximum(K - S_n, 0.0)  # valeur d'exercice immédiat

        # Opérateur de transition du niveau n+1 vers n : taille (n+1, n+2)
        P = build_transition_matrix(n, p, use_sparse=use_sparse)

        # Valeur de continuation (espérance risque-neutre actualisée)
        continuation = discount * (P @ V)

        # Option américaine : max(exercice immédiat, continuation)
        V = np.maximum(exercise_now, continuation)

    # V est de taille 1 à la fin, V[0] = prix à t=0
    return float(V[0])

## 3) Vérification numérique (paramètres demandés)

On teste avec :
\[
S_0=100,\;K=100,\;r=0.05,\;\sigma=0.2,\;T=1,\;N=100.
\]

In [ ]:
# Paramètres de test
S0 = 100
K = 100
r = 0.05
sigma = 0.2
T = 1.0
N = 100

price_am = price_american_put_crr(S0, K, r, sigma, T, N, use_sparse=False)
print(f"Prix du Put américain (CRR, dense) : {price_am:.6f}")

In [ ]:
# Bonus : comparaison dense / sparse (si scipy est disponible)

try:
    price_am_sparse = price_american_put_crr(S0, K, r, sigma, T, N, use_sparse=True)
    print(f"Prix du Put américain (CRR, sparse) : {price_am_sparse:.6f}")
    print(f"Écart absolu dense vs sparse : {abs(price_am - price_am_sparse):.2e}")
except ImportError as e:
    print("Version sparse non testée :", e)

In [ ]:
def price_european_put_crr(S0, K, r, sigma, T, N, use_sparse=False):
    """
    Même schéma CRR mais sans exercice anticipé :
    V_n = e^{-r dt} * P_{n+1} V_{n+1}.
    """
    if N <= 0:
        raise ValueError("N doit être un entier strictement positif.")

    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = np.exp(-sigma * np.sqrt(dt))
    p = (np.exp(r * dt) - d) / (u - d)

    if not (0.0 < p < 1.0):
        raise ValueError("La probabilité risque-neutre p doit vérifier 0 < p < 1.")

    discount = np.exp(-r * dt)
    S_N = stock_prices_level(S0, u, d, N)
    V = np.maximum(K - S_N, 0.0)

    for n in range(N - 1, -1, -1):
        P = build_transition_matrix(n, p, use_sparse=use_sparse)
        V = discount * (P @ V)

    return float(V[0])


price_eu = price_european_put_crr(S0, K, r, sigma, T, N, use_sparse=False)
print(f"Prix du Put européen  (CRR) : {price_eu:.6f}")
print(f"Prix du Put américain (CRR) : {price_am:.6f}")
print(f"V_am - V_eu = {price_am - price_eu:.6f}")

## 4) Commentaire final

On observe bien que le prix du Put américain est supérieur (ou égal) à celui du Put européen :
\[
V^{Am}_0 \ge V^{Eu}_0.
\]
C’est normal, car l’option américaine donne un droit supplémentaire : l’exercice anticipé.

Pour un put, l’exercice avant maturité peut devenir optimal dans certains nœuds (notamment quand le sous-jacent est déjà assez bas), car attendre peut être moins intéressant que verrouiller immédiatement la valeur intrinsèque.

Numériquement, la formule
\[
V_n = \max\!\left(\Phi_n,\;e^{-r\Delta t}P_{n+1}V_{n+1}\right)
\]
résume exactement ce mécanisme :  
- \(e^{-r\Delta t}P_{n+1}V_{n+1}\) = **continuer** ;  
- \(\Phi_n\) = **exercer tout de suite** ;  
- le maximum choisit la meilleure décision locale à chaque nœud.